<a href="https://colab.research.google.com/github/Syed8855/AI_Model_Training/blob/main/randoms_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
print(torch.cuda.is_available())

True


In [2]:
!pip install transformers
!pip install timm
!pip install opencv-python
!pip install matplotlib

In [3]:
import torch
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

from torch.utils.data import Dataset, DataLoader
from transformers import SegformerForSemanticSegmentation
from transformers import SegformerImageProcessor

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
IMAGE_DIR = "/content/drive/MyDrive/Offroad_Segmentation_testImages/Offroad_Segmentation_testImages/Color_Images"

MASK_DIR = "/content/drive/MyDrive/Offroad_Segmentation_testImages/Offroad_Segmentation_testImages/Segmentation"

In [6]:
import os
import shutil
import random

TRAIN_DIR = "/content/drive/MyDrive/Offroad_Segmentation_Training_Dataset/Offroad_Segmentation_Training_Dataset/train"
VAL_DIR = "/content/drive/MyDrive/Offroad_Segmentation_Training_Dataset/Offroad_Segmentation_Training_Dataset/val"

train_img_dir = os.path.join(TRAIN_DIR,"Color_Images")
train_mask_dir = os.path.join(TRAIN_DIR,"Segmentation")

val_img_dir = os.path.join(VAL_DIR,"Color_Images")
val_mask_dir = os.path.join(VAL_DIR,"Segmentation")


# Create folders and clear existing content to ensure fresh state
# Remove existing directories and their contents if they exist
if os.path.exists(val_img_dir):
    shutil.rmtree(val_img_dir)
if os.path.exists(val_mask_dir):
    shutil.rmtree(val_mask_dir)

os.makedirs(val_img_dir)
os.makedirs(val_mask_dir)


# Get matching image-mask pairs
image_files=set(os.listdir(train_img_dir))
mask_files=set(os.listdir(train_mask_dir))

paired_files=list(image_files.intersection(mask_files))


print("Total valid pairs:",len(paired_files))


# Shuffle files
random.shuffle(paired_files)


# 15% validation split
split_size=int(0.15*len(paired_files))
if split_size == 0 and len(paired_files) > 0: # Ensure at least one sample if available
    split_size = 1

val_files=paired_files[:split_size]


for file in val_files:

    src_img=os.path.join(train_img_dir,file)
    dst_img=os.path.join(val_img_dir,file)

    src_mask=os.path.join(train_mask_dir,file)
    dst_mask=os.path.join(val_mask_dir,file)

    # No need for `if not os.path.exists(dst_img):` because we cleared the directories
    shutil.copy(src_img,dst_img)
    shutil.copy(src_mask,dst_mask)


print("Validation samples:",len(val_files))

Total valid pairs: 2857
Validation samples: 428


# Task
Fix the `FileNotFoundError` in the `OffroadDataset` class by modifying its `__init__` method in cell `m34Nttwa2_QE` to correctly filter for image-mask pairs where both files exist, and then re-run the training in the same cell.

## Fix `OffroadDataset` file filtering

### Subtask:
Modify the `__init__` method in the `OffroadDataset` class in cell `m34Nttwa2_QE` to correctly filter for image-mask pairs where both files exist.


**Reasoning**:
The previous run resulted in a `FileNotFoundError` because the `OffroadDataset` class was not correctly filtering for image-mask pairs that exist in both the image and mask directories. The subtask specifically asks to modify the `__init__` method in the `OffroadDataset` class to ensure that `self.files` contains only image names for which both an image and a mask file exist. I will update the code in cell `m34Nttwa2_QE` to implement this fix.



In [ ]:
import torch
import numpy as np
import os
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm import tqdm
import torch.optim as optim

from transformers import SegformerForSemanticSegmentation
from transformers import SegformerImageProcessor

from torch.cuda.amp import autocast, GradScaler


#########################################
# CONFIG
#########################################

NUM_CLASSES = 10
IMAGE_SIZE = 512
BATCH_SIZE = 4
EPOCHS = 10

TRAIN_DIR = "/content/drive/MyDrive/Offroad_Segmentation_Training_Dataset/Offroad_Segmentation_Training_Dataset/train"
VAL_DIR   = "/content/drive/MyDrive/Offroad_Segmentation_Training_Dataset/Offroad_Segmentation_Training_Dataset/val"

SAVE_DIR = "/content/runs"
os.makedirs(SAVE_DIR,exist_ok=True)


#########################################
# MASK CONVERSION
#########################################

def convert_mask(mask):
    # Optimize conversion with cached NumPy Lookup Table (LUT)
    if not hasattr(convert_mask, 'lut'):
        lut = np.zeros(10001, dtype=np.uint8)
        lut[100] = 1
        lut[200] = 2
        lut[300] = 3
        lut[500] = 4
        lut[550] = 5
        lut[700] = 6
        lut[800] = 7
        lut[7100] = 8
        lut[10000] = 9
        convert_mask.lut = lut
    
    arr = np.array(mask).astype(np.int32)
    arr = np.where((arr >= 0) & (arr <= 10000), arr, 0)
    return convert_mask.lut[arr]


#########################################
# DICE LOSS (Optimized)
#########################################

def dice_loss(pred, target, smooth=1e-6):

    pred = torch.softmax(pred, dim=1)

    target_onehot = F.one_hot(target, NUM_CLASSES).permute(0,3,1,2).float()

    intersection = (pred * target_onehot).sum((2,3))
    union = pred.sum((2,3)) + target_onehot.sum((2,3))

    dice = (2*intersection + smooth)/(union + smooth)

    return 1 - dice.mean()


#########################################
# DATASET
#########################################

class OffroadDataset(Dataset):

    def __init__(self,data_dir):

        self.image_dir=os.path.join(data_dir,"Color_Images")
        self.mask_dir=os.path.join(data_dir,"Segmentation")

        # Fix: Filter files to ensure both image and mask exist
        image_files = set(os.listdir(self.image_dir))
        mask_files = set(os.listdir(self.mask_dir))
        self.files = sorted(list(image_files.intersection(mask_files)))

        self.processor = SegformerImageProcessor(
            do_resize=True,
            size=(IMAGE_SIZE,IMAGE_SIZE),
            do_normalize=True
        )

        print(data_dir)
        print("Valid samples:",len(self.files))


    def __len__(self):
        return len(self.files)


    def __getitem__(self,idx):

        name=self.files[idx]

        img_path=os.path.join(self.image_dir,name)
        mask_path=os.path.join(self.mask_dir,name)

        image=Image.open(img_path).convert("RGB")
        mask=Image.open(mask_path)

        mask=convert_mask(mask)

        encoding=self.processor(
            image,
            return_tensors="pt"
        )

        pixel_values=encoding["pixel_values"].squeeze()

        mask=Image.fromarray(mask)
        mask=mask.resize((IMAGE_SIZE,IMAGE_SIZE),Image.NEAREST)

        mask=torch.tensor(np.array(mask),dtype=torch.long)

        return pixel_values,mask


#########################################
# mIoU METRIC
#########################################

def mean_iou(logits,mask):

    pred=torch.argmax(logits,dim=1)

    scores=[]

    for cls in range(NUM_CLASSES):

        inter=((pred==cls)&(mask==cls)).sum().float()
        union=((pred==cls)|(mask==cls)).sum().float()

        if union>0:
            scores.append((inter/union).item())

    return np.mean(scores) if scores else 0


#########################################
# VALIDATION
#########################################

def validate(model,loader,device):

    model.eval()

    ious=[]

    with torch.no_grad():

        for images,masks in loader:

            images=images.to(device,non_blocking=True)
            masks=masks.to(device,non_blocking=True)

            outputs=model(pixel_values=images)

            logits=outputs.logits

            logits=F.interpolate(
                logits,
                size=masks.shape[-2:],
                mode="bilinear",
                align_corners=False
            )

            ious.append(mean_iou(logits,masks))


    return np.mean(ious)


#########################################
# TRAINING
#########################################

def main():

    device=torch.device("cuda" if torch.cuda.is_available() else "cpu")

    print("Device:",device)


    #################################
    # DATASETS
    #################################

    trainset=OffroadDataset(TRAIN_DIR)
    valset=OffroadDataset(VAL_DIR)


    train_loader=DataLoader(
        trainset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=2,
        pin_memory=True,
        persistent_workers=True
    )


    val_loader=DataLoader(
        valset,
        batch_size=BATCH_SIZE,
        pin_memory=True
    )


    #################################
    # MODEL
#################################

    model=SegformerForSemanticSegmentation.from_pretrained(

        "nvidia/segformer-b3-finetuned-ade-512-512",

        num_labels=NUM_CLASSES,

        ignore_mismatched_sizes=True

    )


    model.to(device)


    #################################
    # OPTIMIZER
    #################################

    optimizer=optim.AdamW(
        model.parameters(),

        lr=5e-5
    )


    scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=EPOCHS
    )


    weights=torch.tensor(
        [0.5,3,3,3,3,3,3,3,1,1],
        dtype=torch.float32
    ).to(device)

    criterion=torch.nn.CrossEntropyLoss(weight=weights)


    #################################
    # AMP
    #################################

    scaler=torch.amp.GradScaler('cuda')


    #################################
    # TRAIN LOOP
    #################################

    best_iou=0

    print("\nStarting Training\n")


    for epoch in range(EPOCHS):

        model.train()

        losses=[]

        pbar=tqdm(train_loader)


        for images,masks in pbar:

            images=images.to(device,non_blocking=True)
            masks=masks.to(device,non_blocking=True)


            optimizer.zero_grad()


            with torch.amp.autocast('cuda'):

                outputs=model(pixel_values=images)

                logits=outputs.logits

                logits=F.interpolate(
                    logits,
                    size=masks.shape[-2:],
                    mode="bilinear",
                    align_corners=False
                )

                ce_loss=criterion(logits,masks)

                d_loss=dice_loss(logits,masks)

                loss=ce_loss+0.2*d_loss


            scaler.scale(loss).backward()

            scaler.step(optimizer)

            scaler.update()


            losses.append(loss.item())

            pbar.set_description(
                f"Epoch {epoch+1} Loss {loss.item():.3f}"
            )


        scheduler.step()


        #################################
        # VALIDATION
        #################################

        val_iou=validate(
            model,
            val_loader,
            device
        )


        print("\nEpoch:",epoch+1)
        print("Train Loss:",np.mean(losses))
        print("Val mIoU:",val_iou)


        #################################
        # SAVE BEST MODEL
        #################################

        if val_iou>best_iou:

            best_iou=val_iou

            torch.save(
                model.state_dict(),
                f"{SAVE_DIR}/best_model.pth"
            )

            print("Best model saved")


    print("\nTraining Finished")
    print("Best IoU:",best_iou)

if __name__=="__main__":
    main()


Device: cuda
/content/drive/MyDrive/Offroad_Segmentation_Training_Dataset/Offroad_Segmentation_Training_Dataset/train
Valid samples: 2857
/content/drive/MyDrive/Offroad_Segmentation_Training_Dataset/Offroad_Segmentation_Training_Dataset/val
Valid samples: 428


Loading weights:   0%|          | 0/644 [00:00<?, ?it/s]

SegformerForSemanticSegmentation LOAD REPORT from: nvidia/segformer-b3-finetuned-ade-512-512
Key                           | Status   |                                                                                                    
------------------------------+----------+----------------------------------------------------------------------------------------------------
decode_head.classifier.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([150]) vs model:torch.Size([10])                      
decode_head.classifier.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([150, 768, 1, 1]) vs model:torch.Size([10, 768, 1, 1])

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.



Starting Training



Epoch 1 Loss 0.737:  80%|████████  | 573/715 [04:08<00:56,  2.53it/s]

In [ ]:
import torch
import numpy as np
import os
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import matplotlib.pyplot as plt

from transformers import SegformerForSemanticSegmentation
from transformers import SegformerImageProcessor


#################################
# CONFIG
#################################

NUM_CLASSES = 10
IMAGE_SIZE = 512
BATCH_SIZE = 2

TEST_DIR="/content/drive/MyDrive/3/Offroad_Segmentation_testImages"

MODEL_PATH="/content/runs/best_model.pth"


#################################
# MASK CONVERSION
#################################

def convert_mask(mask):
    # Optimize conversion with cached NumPy Lookup Table (LUT)
    if not hasattr(convert_mask, 'lut'):
        lut = np.zeros(10001, dtype=np.uint8)
        lut[100] = 1
        lut[200] = 2
        lut[300] = 3
        lut[500] = 4
        lut[550] = 5
        lut[700] = 6
        lut[800] = 7
        lut[7100] = 8
        lut[10000] = 9
        convert_mask.lut = lut
    
    arr = np.array(mask).astype(np.int32)
    arr = np.where((arr >= 0) & (arr <= 10000), arr, 0)
    return convert_mask.lut[arr]


#################################
# DATASET
#################################

class TestDataset(Dataset):

    def __init__(self,data_dir):

        self.image_dir=os.path.join(data_dir,"Color_Images")
        self.mask_dir=os.path.join(data_dir,"Segmentation")

        self.files=sorted(os.listdir(self.image_dir))

        self.processor=SegformerImageProcessor(
            do_resize=True,
            size=(IMAGE_SIZE,IMAGE_SIZE),
            do_normalize=True
        )

        print("Test Samples:",len(self.files))


    def __len__(self):

        return len(self.files)


    def __getitem__(self,idx):

        name=self.files[idx]

        img_path=os.path.join(self.image_dir,name)
        mask_path=os.path.join(self.mask_dir,name)

        image=Image.open(img_path).convert("RGB")
        mask=Image.open(mask_path)

        mask=convert_mask(mask)

        encoding=self.processor(
            image,
            return_tensors="pt"
        )

        pixel_values=encoding["pixel_values"].squeeze()

        mask=Image.fromarray(mask)
        mask=mask.resize((IMAGE_SIZE,IMAGE_SIZE),Image.NEAREST)

        mask=torch.tensor(np.array(mask),dtype=torch.long)

        return pixel_values,mask,image


#################################
# mIoU
#################################

def mean_iou(logits,mask):

    pred=torch.argmax(logits,dim=1)

    scores=[]

    for cls in range(NUM_CLASSES):

        inter=((pred==cls)&(mask==cls)).sum().float()
        union=((pred==cls)|(mask==cls)).sum().float()

        if union>0:
            scores.append((inter/union).item())

    return np.mean(scores) if scores else 0


#################################
# MAIN
#################################

def main():

    device=torch.device("cuda" if torch.cuda.is_available() else "cpu")

    print("Device:",device)


    #################################
    # DATA
    #################################

    testset=TestDataset(TEST_DIR)

    test_loader=DataLoader(
        testset,
        batch_size=BATCH_SIZE
    )


    #################################
    # MODEL
    #################################

    model=SegformerForSemanticSegmentation.from_pretrained(

        "nvidia/segformer-b3-finetuned-ade-512-512",

        num_labels=NUM_CLASSES,

        ignore_mismatched_sizes=True
    )


    model.load_state_dict(torch.load(MODEL_PATH))

    model.to(device)

    model.eval()


    #################################
    # EVALUATION
    #################################

    ious=[]

    with torch.no_grad():

        for images,masks,_ in test_loader:

            images=images.to(device)
            masks=masks.to(device)

            outputs=model(pixel_values=images)

            logits=outputs.logits

            logits=F.interpolate(
                logits,
                size=masks.shape[-2:],
                mode="bilinear",
                align_corners=False
            )

            ious.append(mean_iou(logits,masks))


    print("\nTest mIoU:",np.mean(ious))


    #################################
    # VISUALIZE
    #################################

    images,masks,orig=testset[0]

    with torch.no_grad():

        pred=model(pixel_values=images.unsqueeze(0).to(device)).logits

        pred=torch.argmax(pred,1).cpu().squeeze()


    plt.figure(figsize=(12,4))

    plt.subplot(1,3,1)
    plt.title("Image")
    plt.imshow(orig)
    plt.axis("off")


    plt.subplot(1,3,2)
    plt.title("Ground Truth")
    plt.imshow(masks)
    plt.axis("off")


    plt.subplot(1,3,3)
    plt.title("Prediction")
    plt.imshow(pred)
    plt.axis("off")


    plt.show()



if __name__=="__main__":
    main()

In [ ]:
import torch
import numpy as np
import os
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import matplotlib.pyplot as plt

from transformers import SegformerForSemanticSegmentation
from transformers import SegformerImageProcessor


#################################
# CONFIG
#################################

NUM_CLASSES = 10
IMAGE_SIZE = 512
BATCH_SIZE = 2

TEST_DIR=""

MODEL_PATH="/content/runs/best_model.pth"


#################################
# MASK CONVERSION
#################################

def convert_mask(mask):
    # Optimize conversion with cached NumPy Lookup Table (LUT)
    if not hasattr(convert_mask, 'lut'):
        lut = np.zeros(10001, dtype=np.uint8)
        lut[100] = 1
        lut[200] = 2
        lut[300] = 3
        lut[500] = 4
        lut[550] = 5
        lut[700] = 6
        lut[800] = 7
        lut[7100] = 8
        lut[10000] = 9
        convert_mask.lut = lut
    
    arr = np.array(mask).astype(np.int32)
    arr = np.where((arr >= 0) & (arr <= 10000), arr, 0)
    return convert_mask.lut[arr]


#################################
# DATASET
#################################

class TestDataset(Dataset):

    def __init__(self,data_dir):

        self.image_dir=os.path.join(data_dir,"Color_Images")
        self.mask_dir=os.path.join(data_dir,"Segmentation")

        self.files=sorted(os.listdir(self.image_dir))

        self.processor=SegformerImageProcessor(
            do_resize=True,
            size=(IMAGE_SIZE,IMAGE_SIZE),
            do_normalize=True
        )

        print("Test Samples:",len(self.files))


    def __len__(self):

        return len(self.files)


    def __getitem__(self,idx):

        name=self.files[idx]

        img_path=os.path.join(self.image_dir,name)
        mask_path=os.path.join(self.mask_dir,name)

        image=Image.open(img_path).convert("RGB")
        mask=Image.open(mask_path)

        mask=convert_mask(mask)

        encoding=self.processor(
            image,
            return_tensors="pt"
        )

        pixel_values=encoding["pixel_values"].squeeze()

        mask=Image.fromarray(mask)
        mask=mask.resize((IMAGE_SIZE,IMAGE_SIZE),Image.NEAREST)

        mask=torch.tensor(np.array(mask),dtype=torch.long)

        return pixel_values,mask,image


#################################
# mIoU
#################################

def mean_iou(logits,mask):

    pred=torch.argmax(logits,dim=1)

    scores=[]

    for cls in range(NUM_CLASSES):

        inter=((pred==cls)&(mask==cls)).sum().float()
        union=((pred==cls)|(mask==cls)).sum().float()

        if union>0:
            scores.append((inter/union).item())

    return np.mean(scores) if scores else 0


#################################
# MAIN
#################################

def main():

    device=torch.device("cuda" if torch.cuda.is_available() else "cpu")

    print("Device:",device)


    #################################
    # DATA
    #################################

    testset=TestDataset(TEST_DIR)

    test_loader=DataLoader(
        testset,
        batch_size=BATCH_SIZE
    )


    #################################
    # MODEL
    #################################

    model=SegformerForSemanticSegmentation.from_pretrained(

        "nvidia/segformer-b3-finetuned-ade-512-512",

        num_labels=NUM_CLASSES,

        ignore_mismatched_sizes=True
    )


    model.load_state_dict(torch.load(MODEL_PATH))

    model.to(device)

    model.eval()


    #################################
    # EVALUATION
    #################################

    ious=[]

    with torch.no_grad():

        for images,masks,_ in test_loader:

            images=images.to(device)
            masks=masks.to(device)

            outputs=model(pixel_values=images)

            logits=outputs.logits

            logits=F.interpolate(
                logits,
                size=masks.shape[-2:],
                mode="bilinear",
                align_corners=False
            )

            ious.append(mean_iou(logits,masks))


    print("\nTest mIoU:",np.mean(ious))


    #################################
    # VISUALIZE
    #################################

    images,masks,orig=testset[0]

    with torch.no_grad():

        pred=model(pixel_values=images.unsqueeze(0).to(device)).logits

        pred=torch.argmax(pred,1).cpu().squeeze()


    plt.figure(figsize=(12,4))

    plt.subplot(1,3,1)
    plt.title("Image")
    plt.imshow(orig)
    plt.axis("off")


    plt.subplot(1,3,2)
    plt.title("Ground Truth")
    plt.imshow(masks)
    plt.axis("off")


    plt.subplot(1,3,3)
    plt.title("Prediction")
    plt.imshow(pred)
    plt.axis("off")


    plt.show()



if __name__=="__main__":
    main()